In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 21))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
import pandas as pd
from scipy.spatial.distance import cdist


def compute_weights_with_best_rps(
        m_rfp: np.array,
        idx_rfp: np.array,
        m_tp: np.array,
        idx_tp: np.array,
        df_tp: pd.DataFrame = None
) -> (np.array, np.array):
    """
    Computes weights for two matrices with a single reference point parameter.
    :param m_rfp: point matrix for the reference points (2D array)
    :param idx_rfp: valid index matrix for the reference points (1D array)
    :param m_tp: point matrix for the test points (2D array)
    :param idx_tp: valid index matrix for the test points (1D array)
    :return: Weights and sorted indices by weight
    """

    # Compute the Euclidean distances between the TPs and RPs
    D = cdist(m_tp, m_rfp, metric="euclidean")

    # Normalize distances based on common valid indices
    match = np.logical_and(idx_tp[:, np.newaxis, :], idx_rfp[np.newaxis, :, :])
    s = np.sum(match, axis=2)

    # Avoid division by zero by setting distances to a very large value where no matches exist
    realmax = np.finfo(np.float64).max
    D = np.divide(D, s, out=np.full_like(D, realmax), where=s != 0)

    # Set distances to dummy reference points to a very large value
    dummy_rfps = np.all(idx_rfp == 0, axis=1)
    D[:, dummy_rfps] = realmax

    # Replace zero distances with a small value to avoid singularities
    min_nonzero_distance = np.min(D[D > 0])
    D[D == 0] = min_nonzero_distance / 20

    # set non-optimal paths to max value
    if df_tp is not None:
        for i, row in df_tp.iterrows():
            matches = row['matches']
            non_matching_indices = set(range(m_rfp.shape[0])) - set(matches)
            D[i, list(non_matching_indices)] = realmax

    # Sort distances and compute weights
    idx_sort = np.argsort(D, axis=1)
    D_sort = np.take_along_axis(D, idx_sort, axis=1)
    W = 1.0 / D_sort

    return W, idx_sort


In [3]:

from scripts.utils import extract_unique_npcis, dataset_tp_rp_split


#from scripts.beamforming import get_best_beam, find_matching_rps


def find_matching_rps(df_rp: pd.DataFrame, best_beam: np.array):
    mask = df_rp["best_beam"].apply(lambda x: all(x == best_beam))
    return df_rp[mask].index


def find_beam_matches(df_rp: pd.DataFrame, best_beams: np.array) -> list[int]:
    best_beams = np.sort(best_beams, axis=0)

    def compare_beams(beams):
        beams = np.sort(beams, axis=0)
        return np.array_equal(best_beams, beams)

    mask = df_rp["best_beam"].apply(compare_beams)
    return df_rp[mask].index


def get_best_beam(mat: pd.DataFrame, rf_param: RF_PARAM_5G):
    # Check if the DataFrame is empty after dropping NaNs
    if mat.empty:
        print("No valid data available after dropping NaN values.")
        return None, None

    idx = mat[rf_param.value].nlargest(5).index

    return mat.loc[idx][['pci', 'operator_id', 'nr_arfcn', 'beam_index']].values


df = df.sample(2000)

# iterate over tps
errors = []
unique_npcis = extract_unique_npcis(df["measurements_matrix"])

beam_param = RF_PARAM_5G.SINR

df['best_beam'] = df['measurements_matrix'].apply(lambda x: get_best_beam(x, beam_param))

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 42)

df_tp["matches"] = df_tp["best_beam"].apply(lambda x: find_beam_matches(df_rp, x))

df_tp["num_matches"] = df_tp["matches"].apply(lambda x: len(x))

df_tp = df_tp[df_tp["num_matches"] > 5]

df_tp

,lat,lng,measurements_matrix,campaign_id,best_beam,matches,num_matches
3,41.898448,12.429485,pci beam_index nr_arfcn operator_id ...,5,"[[75, 10, 643296, 3], [57, 10, 432050, 3], [-1...","Index([455, 469, 525, 557, 732, 1078, 1166, 13...",8
4,41.896102,12.428702,pci beam_index nr_arfcn operator_id ...,18,"[[76, 10, 643296, 5], [76, 10, 643296, 6], [76...","Index([213, 372, 552, 708, 993, 1007, 1209], d...",7
6,41.898403,12.429538,pci beam_index nr_arfcn operator_id ...,6,"[[57, 10, 432050, 3], [-108, 10, 643296, 5], [...","Index([56, 188, 212, 798, 1230, 1246], dtype='...",6
9,41.895625,12.429365,pci beam_index nr_arfcn operator_id ...,7,"[[-108, 10, 643296, 1], [76, 10, 643296, 5], [...","Index([ 48, 90, 182, 183, 269, 298, 37...",25
10,41.896287,12.428617,pci beam_index nr_arfcn operator_id ...,9,"[[76, 10, 643296, 4], [76, 10, 643296, 5], [76...","Index([163, 173, 334, 340, 379, 407, 561, 647,...",11
...,...,...,...,...,...,...,...
608,41.895669,12.428932,pci beam_index nr_arfcn operator_id ...,1,"[[76, 10, 643296, 5], [76, 10, 643296, 6], [76...","Index([213, 372, 552, 708, 993, 1007, 1209], d...",7
609,41.895503,12.429486,pci beam_index nr_arfcn operator_id ...,10,"[[-108, 10, 643296, 1], [-108, 10, 643296, 0],...","Index([107, 264, 306, 533, 1137, 1223], dtype=...",6
611,41.894818,12.429748,pci beam_index nr_arfcn operator_id ...,3,"[[-108, 10, 643296, 1], [58, 10, 432050, 3], [...","Index([ 33, 53, 60, 84, 96, 121, 13...",45
612,41.898554,12.429622,pci beam_index nr_arfcn operator_id ...,9,"[[-109, 10, 643296, 1], [-108, 10, 643296, 5],...","Index([ 20, 51, 119, 221, 251, 315, 38...",24


In [4]:


from scripts.weighted_coverage import wknn_one
from scripts.matrix_operations import compute_weights, create_point_matrix

m_rfp, idx_rfp = create_point_matrix(df_rp, unique_npcis, rf_param)

m_tp, idx_tp = create_point_matrix(df_tp, unique_npcis, rf_param)

W, idx_sort = compute_weights(m_rfp, idx_rfp, m_tp, idx_tp, df_tp, df_rp)

W_base, idx_sort_base = compute_weights(m_rfp, idx_rfp, m_tp, idx_tp)

_, k_avg_error = wknn_one(df_tp, df_rp, idx_sort, W, 2)

_, k_avg_error_base = wknn_one(df_tp, df_rp, idx_sort_base, W_base, 2)

print(f"""
Mean error = {k_avg_error.mean():.3f}
Mean error (control) = {k_avg_error_base.mean():.3f}
""")

k_avg_error

df_tp['errors'] = k_avg_error

df_tp['errors_base'] = k_avg_error_base

df_tp

matches [213, 372, 552, 708, 993, 1007, 1209]

Mean error = 15.772
Mean error (control) = 7.827



,lat,lng,measurements_matrix,campaign_id,best_beam,matches,num_matches,errors,errors_base
3,41.898448,12.429485,pci beam_index nr_arfcn operator_id ...,5,"[[75, 10, 643296, 3], [57, 10, 432050, 3], [-1...","Index([455, 469, 525, 557, 732, 1078, 1166, 13...",8,36.664590,0.238534
4,41.896102,12.428702,pci beam_index nr_arfcn operator_id ...,18,"[[76, 10, 643296, 5], [76, 10, 643296, 6], [76...","Index([213, 372, 552, 708, 993, 1007, 1209], d...",7,10.082538,0.688875
6,41.898403,12.429538,pci beam_index nr_arfcn operator_id ...,6,"[[57, 10, 432050, 3], [-108, 10, 643296, 5], [...","Index([56, 188, 212, 798, 1230, 1246], dtype='...",6,1.124688,1.163275
9,41.895625,12.429365,pci beam_index nr_arfcn operator_id ...,7,"[[-108, 10, 643296, 1], [76, 10, 643296, 5], [...","Index([ 48, 90, 182, 183, 269, 298, 37...",25,0.287438,0.205348
10,41.896287,12.428617,pci beam_index nr_arfcn operator_id ...,9,"[[76, 10, 643296, 4], [76, 10, 643296, 5], [76...","Index([163, 173, 334, 340, 379, 407, 561, 647,...",11,28.720791,40.875828
...,...,...,...,...,...,...,...,...,...
608,41.895669,12.428932,pci beam_index nr_arfcn operator_id ...,1,"[[76, 10, 643296, 5], [76, 10, 643296, 6], [76...","Index([213, 372, 552, 708, 993, 1007, 1209], d...",7,68.117830,10.155348
609,41.895503,12.429486,pci beam_index nr_arfcn operator_id ...,10,"[[-108, 10, 643296, 1], [-108, 10, 643296, 0],...","Index([107, 264, 306, 533, 1137, 1223], dtype=...",6,7.445563,4.826965
611,41.894818,12.429748,pci beam_index nr_arfcn operator_id ...,3,"[[-108, 10, 643296, 1], [58, 10, 432050, 3], [...","Index([ 33, 53, 60, 84, 96, 121, 13...",45,1.751760,0.058527
612,41.898554,12.429622,pci beam_index nr_arfcn operator_id ...,9,"[[-109, 10, 643296, 1], [-108, 10, 643296, 5],...","Index([ 20, 51, 119, 221, 251, 315, 38...",24,12.488357,12.329294


In [5]:
len(k_avg_error)
len(df_tp)

271

In [13]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def plot_rps(rp_stat_df: pd.DataFrame, tp: pd.Series, title: str):
    plt.figure(figsize=(8, 8))
    ax = plt.gca()  # Get current axes

    # Plot points where best_beam is True (e.g., color='blue')
    plt.scatter(rp_stat_df[rp_stat_df['best_beam'] == True]['lat'], rp_stat_df[rp_stat_df['best_beam'] == True]['lng'],
                color='blue', label='Best Beam', s=100)

    # Plot points where best_beam is False (e.g., color='red')
    plt.scatter(rp_stat_df[rp_stat_df['best_beam'] == False]['lat'],
                rp_stat_df[rp_stat_df['best_beam'] == False]['lng'], color='red', label='Not Best Beam', s=100)

    # plot the test point

    plt.scatter(tp['lat'], tp['lng'], color='green', label='TP', s=100)

    plt.xlabel('Latitude')
    plt.ylabel('Longitude')
    plt.title(title)

    # Force decimal formatting for axis ticks
    formatter = mticker.FormatStrFormatter('%.2f')  # 6 decimal places, adjust as needed
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)
    plt.legend()
    plt.grid(True)
    plt.show()

In [14]:
test = df.iloc[0]

mat = test['measurements_matrix']

idx = mat[rf_param.value].nlargest(3).index

matches = mat.loc[idx][['pci', 'operator_id', 'nr_arfcn', 'beam_index']].values

matches = np.sort(matches, axis=0)
other = matches.copy()

rev = other[::-1]

print(np.array_equal(matches, other))

rev = np.sort(rev, axis=0)

print(np.array_equal(matches, rev))

True
True


In [27]:
from scripts.utils import haversine_distance

indecies = []

tp_dataframes = {}
for i in indecies:
    tp = df_tp.iloc[i]
    tp_sig = m_tp[i, :]
    matches = tp["matches"]

    selected = idx_sort[i, :10]
    selected_base = idx_sort_base[i, :10]
    data = []


    def get_metrics_for_point(index):
        num_pcis = idx_rfp[index, :].sum()
        rp_sig = m_rfp[index, :]
        dist = cdist(tp_sig.reshape(1, -1), rp_sig.reshape(1, -1), metric="euclidean")
        pos_rp = df_rp.loc[index][['lat', 'lng']]

        return num_pcis, rp_sig, dist, pos_rp


    def extract_stats(points, n_range, best_beam):
        for n in range(n_range):
            idx = points[n]
            num_pcis, rp_sig, dist, pos_rp = get_metrics_for_point(idx)
            haversine_dist = haversine_distance(tp['lat'], tp['lng'], pos_rp['lat'], pos_rp['lng'])
            data.append(
                (idx, best_beam, num_pcis, dist[0][0], dist[0][0] / num_pcis, pos_rp['lat'], pos_rp['lng'],
                 haversine_dist))


    extract_stats(selected, 2, True)
    extract_stats(selected_base, 2, False)

    rp_stat_df = pd.DataFrame(data, columns=['idx', 'best_beam', 'num_pcis', 'dist', 'dist_normalized', 'lat', 'lng',
                                             'haversine_dist'])
    tp_dataframes[i] = rp_stat_df
    title = f"'Scatter Plot of Latitude vs. Longitude by Beam Type'\nERROR [Best Beam {tp['errors']:.1f}] [Normal {tp['errors_base']:.1f}]"
    plot_rps(rp_stat_df, tp, title)


good 65 matches
bad 206 matches
sum 271 matches



/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_43970/1839834518.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  good_tps['count'] = good_tps['matches'].apply(lambda x: len(x))
/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_43970/1839834518.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bad_tps['count'] = bad_tps['matches'].apply(lambda x: len(x))


np.float64(21.262135922330096)

In [32]:
good_tps['count'].mean()


np.float64(19.4)